### 33_difhistoryplayer_3 

Script: build_mnm_pipeline.py
* denna notebook [33_difhistoryplayer_3.ipynb](https://github.com/salgo60/ifkdb/blob/main/Notebook/33_difhistoryplayer_3.ipynb)
   * [mnm_update_difplayers_3.csv](https://github.com/salgo60/ifkdb/blob/main/Notebook/mnm_update_difplayers_3.csv)
   * [quickstatements_difplayers_3.txt](https://github.com/salgo60/ifkdb/blob/main/Notebook/quickstatements_difplayers_3.txt)

* tidigare notebook test [33_difhistoryplayer_2.ipynb](https://github.com/salgo60/ifkdb/blob/main/Notebook/33_difhistoryplayer_2.ipynb)

Vibekodad
* 💡 Nästa förbättring som skulle vara väldigt värdefull för DIF-datasetet:
   * extrahera moderklubb
   * extrahera tröjnummer
   * generera P413 (position played on team)

Det går att göra ganska stabilt från samma infobox och ger mycket bättre Wikidata-data.

In [1]:
from datetime import datetime
import platform
import sys
import os

# Start timer
start_time = datetime.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}")


Start time: 2026-03-07 10:36:25


In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from datetime import date

INPUT="players.csv"

TEAM_Q="Q186591"   # Göteborgsalliansen (justera vid behov)

mmupdatefile="mnm_update_goteborgsalliansen.csv"
qsfile="quickstatements_goteborgsalliansen.txt"

df=pd.read_csv(INPUT)

session=requests.Session()

headers={"User-Agent":"goteborgsalliansen-wikidata-bot salgo60@msn.com"}

# --------------------------------------------------
# SCRAPE PLAYER PAGE
# --------------------------------------------------

def scrape(url):

    try:

        r=session.get(url,timeout=10,headers=headers)

        soup=BeautifulSoup(r.text,"html.parser")

        text=soup.get_text("\n",strip=True)

        return text

    except:

        return ""

with ThreadPoolExecutor(max_workers=12) as ex:

    df["page"]=list(tqdm(ex.map(scrape,df["url"]),total=len(df)))

# --------------------------------------------------
# EXTRACT DATA
# --------------------------------------------------

df["born_year"]=df["page"].str.extract(r"Född\s+.*?(\d{4})")

df["position"]=df["page"].str.extract(r"Position\s*([^\n]+)")

df["birthplace"]=df["page"].str.extract(r"Födelseort\s*([^\n]+)")

# --------------------------------------------------
# POSITION MAPPING
# --------------------------------------------------

pos_map={

"målvakt":"Q201330",
"försvarare":"Q230250",
"mittfältare":"Q193592",
"anfallare":"Q280658"

}

df["pos_q"]=df["position"].str.lower().map(pos_map)

# --------------------------------------------------
# MIX'N'MATCH DESCRIPTION
# --------------------------------------------------

df["desc"]="football player"

df.loc[df["born_year"].notna(),"desc"]+=" (born "+df["born_year"]+")"

df.loc[df["club"].notna(),"desc"]+=", "+df["club"]+" player"

df["desc"]=df["desc"].str.replace(r"\s+"," ",regex=True).str.strip()

mnm=df[["player_id","name","desc"]].rename(columns={"player_id":"id"})

mnm["desc"]=mnm["desc"].str.slice(0,150)

mnm.to_csv(mmupdatefile,index=False)

# --------------------------------------------------
# QUICKSTATEMENTS
# --------------------------------------------------

today=date.today().isoformat()

qs=[]

for _,r in df.iterrows():

    q=r.get("qid")

    if pd.isna(q):

        continue

    source=r["url"]

    # birth year
    if pd.notna(r["born_year"]):

        qs.append(

        f'{q}\tP569\t+{r["born_year"]}-00-00T00:00:00Z/9\tS854\t"{source}"\tS813\t+{today}T00:00:00Z/11'

        )

    # position
    if pd.notna(r["pos_q"]):

        qs.append(

        f'{q}\tP413\t{r["pos_q"]}\tS854\t"{source}"\tS813\t+{today}T00:00:00Z/11'

        )

    # membership (club)
    if pd.notna(r["club"]):

        qs.append(

        f'{q}\tP54\t{TEAM_Q}\tS854\t"{source}"\tS813\t+{today}T00:00:00Z/11'

        )

with open(qsfile,"w") as f:

    for line in qs:

        f.write(line+"\n")

# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

print("Pipeline finished")

print("Mix'n'Match file:",mmupdatefile)

print("QuickStatements file:",qsfile)

print("QuickStatements rows:",len(qs))

100%|█████████████████████████████████████████| 395/395 [01:58<00:00,  3.33it/s]

Pipeline finished
Mix'n'Match file: mnm_update_goteborgsalliansen.csv
QuickStatements file: quickstatements_goteborgsalliansen.txt
QuickStatements rows: 0


In [5]:
# End timer
end_time = datetime.now()
duration = end_time - start_time

print("\n--- Execution Summary ---")
print(f"End time:      {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Duration:      {duration}")
print(f"Total seconds: {duration.total_seconds():.2f}")
print(f"Python ver:    {sys.version.split()[0]}")
print(f"Platform:      {platform.system()} {platform.release()}")
print(f"Process ID:    {os.getpid()}")


--- Execution Summary ---
End time:      2026-03-11 16:36:53
Duration:      4 days, 6:00:28.738770
Total seconds: 367228.74
Python ver:    3.12.2
Platform:      Darwin 25.3.0
Process ID:    14543
